## 1. Importar librerías

In [254]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, fbeta_score

## 2. Cargar datasets

In [255]:
test = pd.read_csv("../data/processed/test.csv")

TARGET = "helada_24h"

X_test = test.drop(columns=["fecha_hora", TARGET])
y_test = test[TARGET]

## 3. Cargar modelos entrenados

In [256]:
rf = joblib.load("../models/random_forest.pkl")
svm = joblib.load("../models/svm.pkl")
mlp = joblib.load("../models/neural_network.pkl")

## 4. Probabilidades (Clave del ensamble)

In [257]:
rf_proba = rf.predict_proba(X_test)[:, 1]
svm_proba = svm.predict_proba(X_test)[:, 1]
mlp_proba = mlp.predict_proba(X_test)[:, 1]

## 5. Ensamble Híbrido (Soft voting ponderado)

In [258]:
weights = {
    "rf": 0.50,
    "svm": 0.25,
    "mlp": 0.25
}

y_proba_ensemble = (
    weights["rf"] * rf_proba +
    weights["svm"] * svm_proba +
    weights["mlp"] * mlp_proba
)

## 6. Probar diferentes thresholds

In [259]:
thresholds = np.arange(0.35, 0.55, 0.01)

results = []

for t in thresholds:
    y_pred = (y_proba_ensemble >= t).astype(int)

    results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df

,threshold,accuracy,precision,recall,f1
0,0.35,0.861695,0.542396,0.855626,0.663921
1,0.36,0.865763,0.552301,0.840764,0.666667
2,0.37,0.869153,0.560976,0.830149,0.669521
3,0.38,0.868136,0.560117,0.811040,0.662619
4,0.39,0.867458,0.559524,0.798301,0.657918
5,0.40,0.863729,0.552511,0.770701,0.643617
6,0.41,0.858305,0.541732,0.730361,0.622061
7,0.42,0.854915,0.534400,0.709130,0.609489
8,0.43,0.851186,0.526316,0.679406,0.593142
9,0.44,0.847119,0.516892,0.649682,0.575729


## 7. Elegir mejor threshold

In [260]:
best_row = results_df.sort_values(["f1"], ascending=False).iloc[0]

best_threshold = best_row["threshold"]
best_row

threshold    0.370000
accuracy     0.869153
precision    0.560976
recall       0.830149
f1           0.669521
Name: 2, dtype: float64

## 8. Matriz de confusión final

In [261]:
y_final = (y_proba_ensemble >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_final).ravel()

print("FN:", fn)
print("FP:", fp)
print("TP:", tp)
print("TN:", tn)

FN: 80
FP: 306
TP: 391
TN: 2173


## 9. Guardar predicciones

In [262]:
os.makedirs("../results", exist_ok=True)

pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_final,
    "proba": y_proba_ensemble
})

pred_df.to_csv("../results/predicciones_ensemble.csv", index=False)

## 10. Guardar métricas del ensamble

In [263]:
ensemble_metrics = pd.DataFrame(results)
ensemble_metrics.to_csv("../results/metricas_ensemble.csv", index=False)

ensemble_metrics

,threshold,accuracy,precision,recall,f1
0,0.35,0.861695,0.542396,0.855626,0.663921
1,0.36,0.865763,0.552301,0.840764,0.666667
2,0.37,0.869153,0.560976,0.830149,0.669521
3,0.38,0.868136,0.560117,0.811040,0.662619
4,0.39,0.867458,0.559524,0.798301,0.657918
5,0.40,0.863729,0.552511,0.770701,0.643617
6,0.41,0.858305,0.541732,0.730361,0.622061
7,0.42,0.854915,0.534400,0.709130,0.609489
8,0.43,0.851186,0.526316,0.679406,0.593142
9,0.44,0.847119,0.516892,0.649682,0.575729


## 11. Guardar modelo del ensamble

In [264]:
ensemble_model = {
    "rf": rf,
    "svm": svm,
    "mlp": mlp,
    "weights": weights,
    "threshold": best_threshold
}

joblib.dump(ensemble_model, "../models/ensemble.pkl")

['../models/ensemble.pkl']